# 실습 1 - 실습 데이터 준비

**이미지 추론을 위한 이미지 확보**

이미지 추론을 수행하려면 먼저 **분석 대상 이미지**가 필요하다. 본 실습에서는 이미지를 확보하는 세 가지 방법을 다룬다.

| 방법 | 내용 | 사용 상황 |
|---|---|---|
| ① 직접 업로드 | 보유 이미지를 작업 폴더에 업로드 | 본인 연구 데이터 활용 시 |
| ② URL 입력 | 웹 이미지 주소를 직접 지정 | 준비 없이 즉시 시작 시 |
| ③ Hub 데이터셋 | 공개 데이터셋을 내려받아 사용 | 정답 라벨이 필요한 경우 |

- 세 가지 중 **본인 상황에 맞는 하나만** 수행
- 이후 추론 실습은 확보 방식과 무관하게 동일

---
## 0. 환경 준비

실습에 필요한 라이브러리를 불러온다.

- `PIL` — 이미지를 열고 화면에 표시
- `requests` — 웹에서 이미지를 받아오기
- `datasets` — Hub 데이터셋 내려받기

In [ ]:
from PIL import Image
import requests
from datasets import load_dataset

print("준비 완료")

---
## 1. 방법 ① 직접 업로드

보유한 이미지 파일을 작업 폴더에 올려 사용한다.

### 1-1. 파일 업로드 절차

- ① 좌측 **파일 탐색기**에서 작업 폴더 확인
- ② 상단 **업로드 버튼(↑)** 클릭 또는 파일을 끌어다 놓기
- ③ 목록에 파일이 나타나면 업로드 완료
- ④ 파일 **우클릭 → Copy Path** 로 경로 복사

**주의 사항**

- 파일명은 영문·숫자 권장 (한글·공백은 오류 원인)
- 지원 형식: `jpg`, `png`

### 1-2. 현재 위치 확인

복사한 경로가 어느 폴더 기준인지 확인한다.

- `os.getcwd()` — 현재 작업 폴더의 전체 경로 출력
- `os.listdir(".")` — 현재 폴더의 파일 목록 출력

In [ ]:
import os

print("현재 폴더:", os.getcwd())
print("파일 목록:", os.listdir("."))

### 1-3. 이미지 불러오기

`Image.open()` 으로 파일을 이미지로 불러온다. 마지막 줄에 **변수명만 입력**하면 화면에 이미지가 표시된다.

**경로 형태 세 가지**

- 같은 폴더: `"cat.jpg"`
- 하위 폴더: `"images/cat.jpg"`
- 전체 경로: `"/home/work/SKKU/cat.jpg"`

아래 `image_path` 를 **본인 파일 경로**로 변경한 후 실행한다.

In [ ]:
image_path = "cat.jpg"   # 본인 파일 경로로 변경

image = Image.open(image_path)
image

화면에 이미지가 표시되면 경로가 정상이다.

**오류 발생 시**

- `FileNotFoundError` — 경로·파일명 오타 확인
- 파일 우클릭 → **Copy Path** 로 경로를 다시 복사
- 파일명 대소문자, 확장자(`.jpg` / `.jpeg`) 확인

### 1-4. 이미지 정보 확인

불러온 이미지의 기본 정보를 확인한다.

- `size` — 가로·세로 크기 (픽셀)
- `mode` — 색상 방식 (`RGB` 컬러, `L` 흑백)

대부분의 이미지 모델은 `RGB` 입력을 사용한다.

In [ ]:
print("크기:", image.size)
print("색상 방식:", image.mode)

---
## 2. 방법 ② URL 입력

웹에 있는 이미지의 주소를 파일 경로 대신 직접 지정한다.

### 2-1. 이미지 주소 확보

- 웹 이미지에서 **우클릭 → 이미지 주소 복사**
- 주소가 `.jpg`, `.png` 로 끝나는지 확인
- 페이지 주소가 아닌 **이미지 자체의 주소** 사용

**주의 사항**

- 로그인이 필요한 사이트는 접근 불가
- 사이트에 따라 프로그램 접근이 차단될 수 있음
- 실패 시 방법 ①(직접 업로드) 사용

### 2-2. 주소에서 이미지 불러오기

- `requests.get()` — 주소에서 이미지 데이터 받기
- `stream=True` — 이미지를 데이터 형태로 전달
- `Image.open()` — 받은 데이터를 이미지로 변환

In [ ]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"

image_url = Image.open(requests.get(url, stream=True).raw)
image_url

### 2-3. 추론에 직접 사용

위 코드는 이미지를 **눈으로 확인**하기 위한 용도이다.

실제 추론 시에는 pipeline에 주소를 그대로 입력할 수 있다.

- 예: `clf(url)`
- 별도 내려받기 과정 불필요

---
## 3. 방법 ③ Hub 데이터셋

Hugging Face Hub에 공개된 데이터셋을 내려받아 사용한다.

### 3-1. 데이터셋 찾기

- ① huggingface.co → **Datasets** 탭
- ② 좌측 **Tasks** 필터에서 작업 선택 (Image Classification 등)
- ③ 정렬 기준 선택 (다운로드, 트렌딩, 최신)
- ④ 데이터셋 카드에서 용도·출처·라이선스 확인

**선택 시 확인 사항**

- **미리보기(Data Studio)** — 브라우저에서 내용 직접 확인
- **규모** — 이미지 수, 파일 용량
- **이름 형식** — `제작자/데이터셋명`
- **Use this dataset** 버튼으로 코드 복사 가능

### 3-2. 데이터셋 불러오기

`load_dataset()` 으로 Hub 데이터셋을 내려받는다.

- 첫 인자 — 데이터셋 이름 (`제작자/데이터셋명`)
- `split` — 사용할 구간 지정

**split의 종류**

- `train` — 학습용 구간
- `validation` — 검증용 구간
- `test` — 시험용 구간
- 생략 시 전체 구간을 함께 불러옴

예시로 사용하는 `beans` 는 **콩잎 병해 이미지 3종 분류** 데이터셋이다.

첫 실행은 내려받기로 시간이 소요되며, 이후에는 저장된 파일을 재사용한다.

In [ ]:
ds = load_dataset(
    "AI-Lab-Makerere/beans",
    split="train"
)

### 3-3. 데이터셋 구조 확인

불러온 데이터가 어떤 항목으로 구성되는지 확인한다.

- `features` — 각 항목의 구성 열
- `num_rows` — 전체 이미지 수

열 이름은 데이터셋마다 다르므로(`label`, `labels` 등) **구조 확인은 필수 절차**이다.

In [ ]:
ds

### 3-4. 개별 항목 확인

`ds[0]` 으로 첫 번째 항목을 선택한다. 이미지와 정답이 **함께** 저장된 형태이다.

In [ ]:
ds[0]

### 3-5. 이미지 확인

항목에서 이미지 열만 선택하여 화면에 표시한다.

- `ds[0]["image"]` — 첫 번째 항목의 이미지

In [ ]:
image_ds = ds[0]["image"]
image_ds

### 3-6. 정답 라벨 확인

데이터셋에는 정답이 **번호**로 저장되어 있다. 번호에 대응하는 **이름 목록**을 함께 확인해야 의미를 해석할 수 있다.

- `ds[0]["labels"]` — 정답 분류 번호
- `ds.features["labels"].names` — 번호에 대응하는 이름 목록

In [ ]:
label_id = ds[0]["labels"]
label_names = ds.features["labels"].names

print("정답 번호:", label_id)
print("전체 이름 목록:", label_names)
print("정답 이름:", label_names[label_id])

### 3-7. 여러 이미지 확인

반복문으로 앞부분 항목을 순서대로 확인하여 데이터의 전반적 구성을 파악한다.

In [ ]:
for i in range(3):
    print(i, "-", label_names[ds[i]["labels"]])

ds[1]["image"]

---
## 4. 정리

**세 가지 방법**

- **방법 ①** 직접 업로드 — `Image.open("경로")`
- **방법 ②** URL 입력 — `Image.open(requests.get(url, stream=True).raw)`
- **방법 ③** Hub 데이터셋 — `load_dataset("이름", split="train")`

**공통 사항**

- 화면에 이미지가 표시되면 준비 완료
- pipeline 입력은 **경로, 주소, 이미지 객체** 모두 가능
- 정답 라벨 보유 시 모델 예측 결과와 비교 가능